# Irregular execution

In [ ]:
import matplotlib.pyplot as plt
import time
from multiprocessing import Array, Process

In [ ]:
def fibonacci(n):
	if n == 0:
		return 0
	elif n == 1:
		return 1
	else:
		return fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
values = range(5, 40, 5)

## Secuential version

In [ ]:
times = []

for value in values:
	start = time.time()
	print(f"Value: {value} Fibonacci: {fibonacci(value)}", end = " ")
	elapsed_time = time.time() - start
	times.append(elapsed_time)
	print(f"Elapse time: {elapsed_time}")

x_labels = [str(value) for value in values if value is not None]

plt.bar(x_labels, times, align='center')
plt.ylabel("Elapsed time")
plt.title("Fibonacci")

plt.show()

## Parallel version

In [ ]:
values_length = len(values)
fibonacci_times = Array("d", values_length)
MAX_UNITS = 4
workers = []
workers_times = Array("d", MAX_UNITS)

### Version 1

In [ ]:
worker_values = int((values_length + (MAX_UNITS - values_length % MAX_UNITS)) / MAX_UNITS)

positions_per_worker = [list(range(worker * worker_values, min((worker + 1) * worker_values, values_length))) for worker in range(MAX_UNITS)]


### Version 2

In [ ]:
positions_per_worker = [list(range(worker, values_length, MAX_UNITS)) for worker in range(MAX_UNITS)]

---

In [ ]:
def task(worker, positions):
	start = time.time()
	for position in positions:
		start = time.time()
		fibonacci(values[position])
		end = time.time()

		fibonacci_times[position] = end - start
	end = time.time()

	workers_times[worker] = end - start

for worker in range(0, MAX_UNITS):
	worker = Process(target=task,
					args=(worker, positions_per_worker[worker]))
	workers.append(worker)

start = time.time()
for worker in workers:
	worker.start()

for worker in workers:
	worker.join()
end = time.time()

print(f"Global time: {end - start}")

x_labels = [str(i) for i in range(MAX_UNITS)]

plt.bar(x_labels, workers_times, align='center')
plt.ylabel("Elapsed time")
plt.title("Workers")

plt.show()